# T-RES using DeezyMatch with REL disambiguation

REL disambiguation without filtering out microtoponyms and without adding place of publication.

In [ ]:
import sqlite3
from pathlib import Path

from t_res.geoparser import pipeline, ranking, linking

In [ ]:
# --------------------------------------
# Instantiate the ranker:
ranker = ranking.DeezyMatchRanker(
    resources_path="../resources/",
    mentions_to_wikidata=dict(),
    wikidata_to_mentions=dict(),
    strvar_parameters={
        # Parameters to create the string pair dataset:
        "ocr_threshold": 60,
        "top_threshold": 85,
        "min_len": 5,
        "max_len": 15,
        "w2v_ocr_path": str(Path("../resources/models/w2v/").resolve()),
        "w2v_ocr_model": "w2v_*_news",
        "overwrite_dataset": False,
    },
    deezy_parameters={
        # Paths and filenames of DeezyMatch models and data:
        "dm_path": str(Path("../resources/deezymatch/").resolve()),
        "dm_cands": "wkdtalts",
        "dm_model": "w2v_ocr",
        "dm_output": "deezymatch_on_the_fly",
        # Ranking measures:
        "ranking_metric": "faiss",
        "selection_threshold": 25,
        "num_candidates": 3,
        "search_size": 3,
        "verbose": False,
        # DeezyMatch training:
        "overwrite_training": False,
        "do_test": False,
    },
)

In [ ]:
with sqlite3.connect("../resources/rel_db/embeddings_database.db") as conn:
    cursor = conn.cursor()
    linker = linking.RelDisambLinker(
        resources_path="../resources/",
        ranker=ranker,
        linking_resources=dict(),
        rel_params={
            "model_path": "../resources/models/disambiguation/",
            "data_path": "../tests/sample_files/experiments/outputs/data/lwm/",
            "training_split": "originalsplit",
            "context_length": 100,
            "db_embeddings": cursor,
            "with_publication": False,
            "without_microtoponyms": False,
            "do_test": False,
            "default_publname": "",
            "default_publwqid": "",
        },
        overwrite_training=False,
)

In [ ]:
geoparser = pipeline.Pipeline(ranker=ranker, linker=linker)

In [ ]:
predictions = geoparser.run("A remarkable case of rattening has just occurred in the building trade at Shefrield, but also in Lancaster. Not in Nottingham though. Not in Ashton either, nor in Salop!")
print(predictions)

In [ ]:
predictions = geoparser.run("A remarkable case of rattening has just occurred in the building trade at Sheffield.")
print(predictions)